<a href="https://colab.research.google.com/github/AlanAmaro13/3d-classification/blob/main/0_DataGeneration_PaleoVox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generate 3D Cubes and Spheres

## Used libraries

PaleoVox requieres Open3D.

In [ ]:
! pip install open3d

Let's import PaleoVox to perform the data augmentation.

In [ ]:
import sys
import os

# Add the AmaroXI folder to Python path
sys.path.append('/content/drive/MyDrive/cellularAutomata/3dClassification')

In [ ]:
from paleovox import *

In [ ]:
import numpy as np
import math

## Data Generation

The data considered to perform the 3D classification corresponds to a Cube and a Sphere.

In [ ]:
def create_sphere(voxel_size=32, radius=12, center=None):
    """Create a binary sphere in a 3D voxel grid"""
    if center is None:
        center = (voxel_size//2, voxel_size//2, voxel_size//2)

    # Create coordinate grids
    x = np.arange(voxel_size)
    y = np.arange(voxel_size)
    z = np.arange(voxel_size)
    xx, yy, zz = np.meshgrid(x, y, z, indexing='ij')

    # Calculate distance from center
    distance = np.sqrt((xx - center[0])**2 +
                      (yy - center[1])**2 +
                      (zz - center[2])**2)

    # Create binary sphere
    sphere = (distance <= radius).astype(int)
    return sphere



In [ ]:
_sphere = create_sphere() # We use the default parameters.

In [ ]:
plot_voxels(_sphere)

And let's consider the cube.

In [ ]:
def create_cube(voxel_size=32, cube_size=16, center=None):
    """Create a binary cube in a 3D voxel grid"""
    if center is None:
        center = (voxel_size//2, voxel_size//2, voxel_size//2)

    # Calculate half size
    half_size = cube_size // 2

    # Create coordinate grids
    x = np.arange(voxel_size)
    y = np.arange(voxel_size)
    z = np.arange(voxel_size)
    xx, yy, zz = np.meshgrid(x, y, z, indexing='ij')

    # Create binary cube
    cube = ((xx >= center[0] - half_size) & (xx <= center[0] + half_size) &
            (yy >= center[1] - half_size) & (yy <= center[1] + half_size) &
            (zz >= center[2] - half_size) & (zz <= center[2] + half_size)).astype(int)
    return cube

In [ ]:
_cube = create_cube() # We use the define parameters.

In [ ]:
plot_voxels(_cube)

## Visualization of PaleoVox over the Sphere and Cube.

Let's consider the erotion function:

In [ ]:
plot_voxels(
    erotion_general(
      voxel = _sphere,
      axis_idx = 0,
      increment_min = 0.5,
      pr = True
    )
)

Axis: 0
Max value: 28 - Min value: 5 - Coefficient: 5.60 - Restriction: 14
Cut point: 22


Let's consider the deformation function:


In [ ]:
plot_voxels(
    deformation(
        voxel_array = _sphere,
        compaction_factor = 0.6,
        compaction_axis = 0,
    )
)

Now let's rotate the cube 45 degrees in all the axis

In [ ]:
plot_voxels(
    rotate_voxel(
        _cube,
        math.radians(45),
        math.radians(45),
        math.radians(45)
    )
)

For the _propagator fracture_ we need to implement two modifications. These modifications are done because PaleoVox in its version 1.0.5 doesn't have support for different size of voxels, just for 128x128x128.

These new set of fuction does support different voxel sizes.

In [ ]:
def null_planes(voxel_curve, axis):
  empty = create_voxel_grid(voxel_curve.shape[0])
  coords = np.argwhere(voxel_curve == 1 ) # (n,3)

  if axis == 0:
    empty[ :, coords[:,1], coords[:,2] ] = 1

  elif axis == 1:
    empty[ coords[:,0], :, coords[:,2] ]= 1

  elif axis == 2:
    empty[ coords[:,0], coords[:,1], : ] = 1

  else:
    print('Invalid axis!')

  return empty

In [ ]:
def propagator_fracture(voxel_grid, max_position:int=10, return_both: bool = False, pr:bool = False):
    """
    Generate synthetic fracture patterns within a 3D voxel volume using stochastic propagation.

    This function simulates the growth of fractures or cracks through a solid object by
    iteratively propagating from an initial point through available space. It creates
    natural-looking fracture networks by randomly selecting propagation paths from
    nearby voxels, producing patterns reminiscent of cracks in materials, geological
    fractures, or biological branching structures.

    The algorithm works through the following steps:

    1. **Initialization**: Creates a copy of the original voxel grid and an empty grid
       for the fracture pattern.

    2. **Axis Selection**: Identifies the dominant axis of the object by finding which
       dimension (x, y, or z) has the smallest sum of occupied coordinates. This helps
       orient fracture propagation relative to the object's shape.

    3. **Voxel Sorting**: Sorts all occupied voxels lexicographically (by z, then y,
       then x) to create an ordered traversal path through the volume.

    4. **Starting Point Selection**: Chooses an initial point closest to the origin
       using the `get_closest_to_origin` function.

    5. **Iterative Propagation**: Repeatedly:
       - Identifies all voxels that are "ahead" of the current point in the sorted order
       - Calculates distances to these candidate voxels
       - Randomly selects one of the nearest `max_position` candidates as the next point
       - Draws a line between current and next point using Bresenham's algorithm
       - Updates the current point to the newly selected voxel

    6. **Termination**: Stops when no more voxels are available ahead of the current point.

    7. **Output**: Returns either the fracture pattern alone or both the original object
       with fractures removed and the fracture pattern separately.

    Parameters
    ----------
    voxel_grid : numpy.ndarray
        3D binary array representing the original solid object. Values of 1 indicate
        solid material, 0 indicates empty space. The fracture will propagate through
        the solid region.

    max_position : int, default=10
        Maximum number of nearest candidates to consider for the next propagation step.
        A random point among the `max_position` closest available voxels is selected.
        Higher values create more random, wandering fractures; lower values create
        more directed, straight fractures.

    return_both : bool, default=False
        If False, returns only the fractured object (original minus fracture lines).
        If True, returns a list containing:
        - [0]: The fractured object (original minus fracture lines)
        - [1]: The fracture pattern itself (just the lines)

    pr : bool, default=False
        If True, prints debug information including:
        - Shape of occupied voxels
        - Selected dominant axis
        - First few sorted voxels
        - Initial point
        - Available space dimensions
        - Selected next points during propagation

    Returns
    -------
    numpy.ndarray or list
        If return_both=False: 3D binary array of the fractured object (original with
        fracture lines removed).

        If return_both=True: List with two elements:
        - fractured_object : numpy.ndarray
            Original voxel grid with fracture lines removed
        - fracture_pattern : numpy.ndarray
            Binary array containing only the fracture lines

    Notes
    -----
    **Algorithm Details:**

    1. **Axis Selection Logic**:
       The dominant axis is found by `np.argmin(np.sum(on, axis=0))`. This identifies
       the dimension with the smallest total coordinate sum, which typically corresponds
       to the shortest axis of the object. This information is used later in the
       `null_planes` function to clean up the fracture pattern.

    2. **Lexicographic Sorting**:
       `np.lexsort((on[:, 2], on[:, 1], on[:, 0]))` sorts voxels first by z-coordinate,
       then by y, then by x. This creates a consistent traversal order through the
       volume, ensuring that propagation always moves "forward" in this ordering.

    3. **Propagation Constraints**:
       The condition `np.all(a < on_sorted, axis=1)` ensures that propagation only
       considers voxels that are greater than the current point in ALL coordinates.
       This creates a directional bias and prevents backtracking.

    4. **Stochastic Selection**:
       The random selection from the nearest `max_position` candidates creates
       natural variation in fracture paths while maintaining overall directionality.

    5. **Line Drawing**:
       Uses `add_line_to_voxel` (Bresenham's 3D algorithm) to create continuous
       fracture lines between propagation points.

    **Conceptual Model:**

    The algorithm models fracture propagation as:
    - Fractures start at a point of weakness (closest to origin)
    - They propagate through the material following paths of least resistance
    - Random variations create natural-looking crack patterns
    - Fractures never intersect themselves (due to forward-only constraint)
    - Multiple fracture segments connect to form a network

    **Applications:**

    - Simulating cracks in damaged fossils or bones
    - Generating training data for fracture detection algorithms
    - Modeling geological joint patterns in rock
    - Creating synthetic damage patterns for material science
    - Generating branching structures (with parameter tuning)
    - Simulating excavation or mining paths

    Examples
    --------
    Basic fracture generation:

    >>> import numpy as np
    >>>
    >>> # Load or create a voxelized object
    >>> fossil = mesh_to_voxel("trilobite.obj", dimensions=128)
    >>> print(f"Original solid voxels: {np.sum(fossil)}")
    Original solid voxels: 45231
    >>>
    >>> # Generate a fracture pattern
    >>> fractured = propagator_fracture(fossil, max_position=10, pr=True)
    on shape: (45231, 3)
    axis: 2
    The sorted elements are: [[ 0  0  0]
     [ 0  0  1]
     [ 0  0  2]
     [ 0  0  3]
     [ 0  0  4]]
    initial point: [0 0 0]
    The available space is: (45230, 3)
    [1 1 1]
    ...
    >>> print(f"Fractured solid voxels: {np.sum(fractured)}")
    Fractured solid voxels: 44897

    Getting both fracture pattern and fractured object:

    >>> result = propagator_fracture(fossil, max_position=15, return_both=True)
    >>> fractured_obj = result[0]  # Original with fractures removed
    >>> fracture_lines = result[1]  # Just the fracture pattern

    Technical Details
    -----------------
    - **Complexity**: O(N log N) for sorting + O(M) for propagation, where N is
      number of occupied voxels and M is number of fracture segments
    - **Memory**: Creates copies of voxel grids (~2-3x memory overhead)
    - **Randomness**: Uses Python's random module (import random) - ensure seed
      is set for reproducibility
    - **Termination**: Stops when no voxels remain ahead of current point

    **Dependencies on External Functions:**

    This function relies on several external functions that must be defined:
    - `create_voxel_grid()`: Creates an empty voxel grid of appropriate size
    - `get_closest_to_origin()`: Finds the voxel closest to origin (0,0,0)
    - `add_line_to_voxel()`: Draws 3D lines using Bresenham's algorithm
    - `null_planes()`: Post-processes fracture pattern

    Requirements
    ------------
    - numpy (imported as np)
    - random (standard library)
    - External functions: create_voxel_grid, get_closest_to_origin,
      add_line_to_voxel, null_planes

    See Also
    --------
    add_line_to_voxel : Draws individual fracture segments
    erotion_general : Random erosion for surface damage
    deformation : Shape deformation for compaction simulation

    Warnings
    --------
    - **External Dependencies**: Requires several external functions to be defined.
      Ensure `create_voxel_grid`, `get_closest_to_origin`, and `null_planes` exist.

    - **Termination Condition**: The while loop may become infinite if the propagation
      gets stuck. The try-except catches errors but may mask underlying issues.

    - **Directional Bias**: The `np.all(a < on_sorted, axis=1)` constraint forces
      propagation to always move to points with all coordinates larger than current.
      This creates a strong directional bias that may not match all fracture types.

    - **Starting Point**: Always starts at the point closest to origin. For objects
      not located near origin, fractures may initiate at the object's edge.

    - **Memory Usage**: Creates multiple copies of voxel grids (`v_original`, `v`,
      and potentially return copies). For large grids (256³+), this can exceed memory.

    - **Random Behavior**: Results vary between runs unless random seed is fixed.
      Use `random.seed(42)` before calling for reproducible results.

    - **Error Handling**: The bare `except Exception as e` catches all exceptions,
      which may hide important errors.

    - **Return Behavior**: When an exception occurs, the function returns based on
      `return_both` without completing the fracture. This may produce partial results.

    - **No Max Iterations**: The while loop has no maximum iteration guard, which
      could potentially run very long for large volumes.
    """
    v_original = voxel_grid.copy() # This is a copy of the main fossile
    v = create_voxel_grid(size = voxel_grid.shape[0]) # This is empty

    on = np.argwhere(v_original == 1) # (n,3)
    if pr: print('on shape:', on.shape)

    _axis = np.argmin(np.sum(on, axis = 0))
    if pr: print('axis:', _axis)


    on_sorted = on[
        np.lexsort(
            (on[:, 2], on[:, 1], on[:, 0]) # This is our available space organized
            )
        ]
    if pr: print('The sorted elements are:', on_sorted[:5])

    a = get_closest_to_origin(on_sorted) # Get a initial point
    if pr: print('initial point:', a)

    # Start iterating the space

    while True:
        try:

            #print(a)
            mask = np.all(a < on_sorted, axis=1)
            available_space = on_sorted[mask]
            if pr: print('The availanle space is:', available_space.shape)

            #max_distance = np.sqrt( np.sum((available_space[-1] - available_space[0] )**2 ) )
            #min_distance = np.sqrt( np.sum((available_space[1] - available_space[0] )**2 ) )

            differences = available_space - a  # Broadcasting to (n, 3)
            distances = np.sqrt(np.sum(differences**2, axis=1))

            indices = np.argsort(distances)

            b = available_space[indices[ random.randint(0, max_position) ]]
            if pr: print(b)


            #print(distances[indices[position]] )

            add_line_to_voxel(v, a, b) # This will draw a line from a to b

            a = b #

        except Exception as e:

            _final = v_original - null_planes(v, _axis)
            _ind255 = np.where(_final == 255)
            _final[_ind255] = 0

            #print(e)
            if return_both:
                return [
                    _final,
                    null_planes(v, _axis)
                    ]
            elif return_both == False:
                return _final


And let's fracture the sphere:

In [ ]:
_fractured = propagator_fracture(
        voxel_grid = _sphere,
        max_position = 10,
        return_both = True,
        pr = True
)

on shape: (7153, 3)
axis: 0
The sorted elements are: [[ 4 16 16]
 [ 5 12 14]
 [ 5 12 15]
 [ 5 12 16]
 [ 5 12 17]]
initial point: [ 9  9 10]
The availanle space is: (4247, 3)
[10 11 12]
The availanle space is: (3088, 3)
[11 12 14]
The availanle space is: (2260, 3)
[12 15 15]
The availanle space is: (1428, 3)
[14 17 16]
The availanle space is: (814, 3)
[15 18 18]
The availanle space is: (448, 3)
[17 19 20]
The availanle space is: (166, 3)
[19 20 22]
The availanle space is: (30, 3)
[20 23 23]
The availanle space is: (0, 3)


In [ ]:
plot_voxels(
   _fractured[0],
   _fractured[1]
)


## Main Data Generation

Let's create our pipeline. We consider the deformation in all the axis, the propagation of a fracture, a rotation, another fracture, another rotation and another fracture, finally we apply an erotion.


In [ ]:
def pipeline(voxel):

  augmented = voxel.copy()

  augmented = deformation(
      augmented,
      compaction_factor = np.random.uniform(0.6, 0.95),
      compaction_axis = np.random.randint(3)
  )

  augmented = propagator_fracture(
      augmented
  )

  augmented = rotate_voxel(
      augmented,
      math.radians(np.random.uniform(0, 360)),
      math.radians(np.random.uniform(0, 360)),
      math.radians(np.random.uniform(0,360) )
  )

  augmented = propagator_fracture(
      augmented
  )

  augmented = rotate_voxel(
      augmented,
      math.radians(np.random.uniform(0, 360)),
      math.radians(np.random.uniform(0, 360)),
      math.radians(np.random.uniform(0,360) )
  )

  augmented = propagator_fracture(
      augmented
  )

  augmented = erotion_general(
      augmented,
      axis_idx = np.random.randint(3),
      increment_min = 0.75
      )

  return augmented

Let's visualize this pipeline over the cube and the sphere.

In [ ]:
plot_voxels(
    pipeline(_cube)
)

In [ ]:
plot_voxels(
    pipeline(_sphere)
)

## Data Augmentation

Let's simulate 10k samples of each type:

In [ ]:
spheres_list = []
cubes_list = []
for i in range(10000):
  spheres_list.append(pipeline(_sphere))
  cubes_list.append(pipeline(_cube))

In [ ]:
spheres_list = np.array(spheres_list)
cubes_list = np.array(cubes_list)

In [ ]:
spheres_list.shape, cubes_list.shape

((10000, 32, 32, 32), (10000, 32, 32, 32))

In [ ]:
np.save('/content/drive/MyDrive/cellularAutomata/3dClassification/database/spheres', spheres_list)
np.save('/content/drive/MyDrive/cellularAutomata/3dClassification/database/cubes', cubes_list)